# Multi-Asset Momentum Rotation Strategy
## Master Control Panel & Strategy Optimizer

In [ ]:
# =============================================================================
# CELL 1 - CONFIGURATION (Master Control Panel)
# =============================================================================

# --- Library Imports ---
import pandas as pd
import vectorbt as vbt
import numpy as np
from tqdm import tqdm
import warnings
import matplotlib.pyplot as plt
import yfinance as yf

warnings.filterwarnings('ignore')

# =============================================================================
# MOMENTUM ROTATION SETTINGS
# =============================================================================

CASH_THRESHOLD = 40  # Minimum momentum score to enter positions (0-100)
MOMENTUM_LOOKBACK = 1  # Days to average momentum score (1 = no smoothing)

# Momentum scoring weights (must sum to 1.0)
TREND_WEIGHT = 0.4
VOLUME_WEIGHT = 0.3
PRICE_MOMENTUM_WEIGHT = 0.3

# Assets to trade
ASSETS = ['TQQQ', 'GLD', 'BTC-USD']
ASSET_NAMES = ['TQQQ', 'GLD', 'BTC']  # Display names

# =============================================================================
# FEATURE FLAGS (Enable/Disable Indicators)
# Set to True to include in optimization, False to exclude
# =============================================================================

USE_VOLUME_FILTER = True
USE_RSI = False  # Disabled for simplified momentum rotation
USE_STOCH_RSI = False
USE_AROON = False
USE_BOLLINGER = False
USE_CCI = False
USE_SCHAFF_TREND = False
USE_KAMA = False
USE_ALMA = False
USE_MACD = False
USE_ADX = False
USE_OBV = False
USE_CMF = False
USE_ATR = False
USE_BB_WIDTH = False

# =============================================================================
# BASE STRATEGY PARAMETERS (Always Active - Triple EMA Crossover)
# =============================================================================

fast_range = range(4, 20, 4)     # Fast EMA periods: 4, 8, 12, 16
med_range = range(50, 90, 10)     # Medium EMA periods: 50, 60, 70, 80
slow_range = range(120, 250, 20)  # Slow EMA periods: 120, 140, ... 240

# =============================================================================
# INDICATOR PARAMETER RANGES
# =============================================================================

# --- VOLUME INDICATORS ---
vol_ma_range = range(10, 50, 10)       # Volume MA period: 10, 20, 30, 40
vol_mult_range = np.arange(0.8, 1.4, 0.2)  # Volume multiplier threshold

# --- RSI (disabled by default) ---
rsi_window_range = range(7, 22, 7)
rsi_threshold_range = [40, 45, 50, 55]

# =============================================================================
# DATA & SPLIT SETTINGS
# =============================================================================

TRAIN_SPLIT_RATIO = 0.6  # 60% train, 40% validation
DATA_START_DATE = '2017-01-01'  # Start date for yfinance data

# =============================================================================
# BACKTEST SETTINGS
# =============================================================================

batch_size = 100       # Number of strategies per batch (memory management)
init_cash = 100_000    # Initial portfolio value
fees = 0.0005          # Trading fees (0.05%)

# =============================================================================
# CONFIGURATION SUMMARY
# =============================================================================

# Collect active filters
active_filters = []
if USE_VOLUME_FILTER: active_filters.append('VOLUME_FILTER')
if USE_RSI: active_filters.append('RSI')

# Count total parameter ranges
param_count = 3  # Base EMA ranges
if USE_VOLUME_FILTER: param_count += 2
if USE_RSI: param_count += 2

print("✅ Configuration loaded")
print(f"Assets: {ASSET_NAMES}")
print(f"Active filters: {active_filters if active_filters else ['None (Pure EMA Strategy)']}")
print(f"Total parameter ranges defined: {param_count}")
print(f"Cash threshold: {CASH_THRESHOLD}")
print(f"Momentum weights: Trend={TREND_WEIGHT}, Volume={VOLUME_WEIGHT}, Price={PRICE_MOMENTUM_WEIGHT}")

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def extract_best_params(params_tuple):
    """Extract best parameters from tuple into a dictionary."""
    p = {}
    p['fast_ema'] = params_tuple[0]
    p['med_ema'] = params_tuple[1]
    p['slow_ema'] = params_tuple[2]
    idx = 3
    
    if USE_VOLUME_FILTER:
        p['vol_window'] = params_tuple[idx]
        p['vol_mult'] = params_tuple[idx + 1]
        idx += 2
    if USE_RSI:
        p['rsi_window'] = params_tuple[idx]
        p['rsi_threshold'] = params_tuple[idx + 1]
        idx += 2
    return p

def calculate_momentum_score(close_data, volume_data, params):
    """
    Calculate momentum score for an asset.
    Returns a score between 0-100 based on:
    - Trend alignment (EMA alignment)
    - Volume strength
    - Price momentum (fast vs slow EMA distance)
    """
    # Calculate EMAs
    ema_f = vbt.MA.run(close_data, window=params['fast_ema'], ewm=True).ma.squeeze()
    ema_m = vbt.MA.run(close_data, window=params['med_ema'], ewm=True).ma.squeeze()
    ema_s = vbt.MA.run(close_data, window=params['slow_ema'], ewm=True).ma.squeeze()
    
    # 1. Trend Alignment Score (0-100)
    perfect_uptrend = (ema_f > ema_m) & (ema_m > ema_s)
    partial_uptrend = (ema_f > ema_m) | (ema_m > ema_s)
    trend_score = pd.Series(0.0, index=close_data.index)
    trend_score[partial_uptrend] = 50.0
    trend_score[perfect_uptrend] = 100.0
    
    # 2. Volume Strength Score (0-100)
    vol_ma = vbt.MA.run(volume_data, window=params['vol_window']).ma.squeeze()
    vol_ratio = (volume_data / vol_ma - 1) * 100
    volume_score = vol_ratio.clip(lower=0, upper=100)
    
    # 3. Price Momentum Score (0-100)
    sensitivity_factor = 10
    price_momentum = ((ema_f / ema_s - 1) * 100) * sensitivity_factor
    price_momentum_score = price_momentum.clip(lower=0, upper=100)
    
    # Combined momentum score
    momentum_score = (
        trend_score * TREND_WEIGHT +
        volume_score * VOLUME_WEIGHT +
        price_momentum_score * PRICE_MOMENTUM_WEIGHT
    )
    
    return momentum_score, {'trend': trend_score, 'volume': volume_score, 'price': price_momentum_score}

print("\n✅ Helper functions loaded")

In [ ]:
# =============================================================================
# CELL 2 - DATA LOADING (Multi-Asset via yfinance)
# =============================================================================

print("📥 Loading data for assets:", ASSET_NAMES)
print("=" * 50)

# Download data for all assets from yfinance
asset_data = {}

for ticker, name in zip(ASSETS, ASSET_NAMES):
    print(f"Downloading {name} ({ticker})...")
    data = yf.download(ticker, start=DATA_START_DATE, progress=False)
    
    # Handle MultiIndex columns if present
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    
    asset_data[name] = {
        'close': data['Close'].astype(float),
        'volume': data['Volume'].astype(float),
        'high': data['High'].astype(float),
        'low': data['Low'].astype(float),
        'data': data
    }
    print(f"  → {len(data)} samples from {data.index[0].date()} to {data.index[-1].date()}")

# Find common date range across all assets
common_start = max(asset_data[name]['close'].index[0] for name in ASSET_NAMES)
common_end = min(asset_data[name]['close'].index[-1] for name in ASSET_NAMES)

print()
print(f"Common date range: {common_start.date()} to {common_end.date()}")

# Align all assets to common date range
for name in ASSET_NAMES:
    mask = (asset_data[name]['close'].index >= common_start) & (asset_data[name]['close'].index <= common_end)
    asset_data[name]['close'] = asset_data[name]['close'][mask]
    asset_data[name]['volume'] = asset_data[name]['volume'][mask]
    asset_data[name]['high'] = asset_data[name]['high'][mask]
    asset_data[name]['low'] = asset_data[name]['low'][mask]

# Use first asset for reference date index
reference_close = asset_data[ASSET_NAMES[0]]['close']

# Train/Validation split
split_idx = int(len(reference_close) * TRAIN_SPLIT_RATIO)
split_date = reference_close.index[split_idx]

# Split data for each asset
train_data = {}
val_data = {}

for name in ASSET_NAMES:
    train_data[name] = {
        'close': asset_data[name]['close'].iloc[:split_idx],
        'volume': asset_data[name]['volume'].iloc[:split_idx],
        'high': asset_data[name]['high'].iloc[:split_idx],
        'low': asset_data[name]['low'].iloc[:split_idx]
    }
    val_data[name] = {
        'close': asset_data[name]['close'].iloc[split_idx:],
        'volume': asset_data[name]['volume'].iloc[split_idx:],
        'high': asset_data[name]['high'].iloc[split_idx:],
        'low': asset_data[name]['low'].iloc[split_idx:]
    }

# Print data summary
print()
print("📊 DATA SUMMARY")
print("=" * 50)
print(f"Total samples (aligned): {len(reference_close):,}")
print()
print(f"Training period: {train_data[ASSET_NAMES[0]]['close'].index[0].date()} to {train_data[ASSET_NAMES[0]]['close'].index[-1].date()}")
print(f"Training samples: {len(train_data[ASSET_NAMES[0]]['close']):,} ({TRAIN_SPLIT_RATIO*100:.0f}%)")
print()
print(f"Validation period: {val_data[ASSET_NAMES[0]]['close'].index[0].date()} to {val_data[ASSET_NAMES[0]]['close'].index[-1].date()}")
print(f"Validation samples: {len(val_data[ASSET_NAMES[0]]['close']):,} ({(1-TRAIN_SPLIT_RATIO)*100:.0f}%)")
print()
print("✅ Multi-asset data loaded successfully")

In [ ]:
# =============================================================================
# CELL 3 - BUILD COMBINATIONS
# =============================================================================

# Start with base EMA combinations (fast < medium < slow constraint)
combos = [
    (f, m, s) 
    for f in fast_range 
    for m in med_range 
    for s in slow_range 
    if f < m < s
]

print(f"Base EMA combinations: {len(combos):,}")

# Expand combinations based on enabled flags
if USE_VOLUME_FILTER:
    combos = [(*c, v_w, v_m) for c in combos for v_w in vol_ma_range for v_m in vol_mult_range]
    print(f"+ Volume Filter: {len(combos):,} combinations")

if USE_RSI:
    combos = [(*c, rsi_w, rsi_thresh) for c in combos 
              for rsi_w in rsi_window_range 
              for rsi_thresh in rsi_threshold_range]
    print(f"+ RSI: {len(combos):,} combinations")

# Count active filters
count_active = len(active_filters)

print()
print("=" * 50)
print(f"🔬 Testing {len(combos):,} strategies per asset with {count_active} active filters")
print(f"Total strategies to test: {len(combos) * len(ASSET_NAMES):,} across {len(ASSET_NAMES)} assets")

In [ ]:
# =============================================================================
# CELL 4 - MULTI-ASSET GRID SEARCH
# =============================================================================

# Storage for best parameters per asset
best_params_per_asset = {}

print("🔍 Running Grid Search for Each Asset")
print("=" * 50)

for asset_name in ASSET_NAMES:
    print(f"\n📈 Asset: {asset_name}")
    print("-" * 40)
    
    # Get training data for this asset
    train_close = train_data[asset_name]['close']
    train_volume = train_data[asset_name]['volume']
    train_high = train_data[asset_name]['high']
    train_low = train_data[asset_name]['low']
    
    # Unpack combinations into separate parameter lists
    unpacked = list(zip(*combos))
    fast_periods = list(unpacked[0])
    med_periods = list(unpacked[1])
    slow_periods = list(unpacked[2])
    
    idx = 3
    if USE_VOLUME_FILTER:
        vol_windows = list(unpacked[idx])
        vol_mults = list(unpacked[idx + 1])
        idx += 2
    if USE_RSI:
        rsi_windows = list(unpacked[idx])
        rsi_thresholds = list(unpacked[idx + 1])
        idx += 2
    
    # Initialize result storage for this asset
    all_sharpe = []
    all_returns = []
    all_combos = []
    
    # Process in batches for memory efficiency
    for i in tqdm(range(0, len(combos), batch_size), desc=f"{asset_name} Progress", leave=False):
        
        # Extract batch slice
        batch_combos = combos[i:i + batch_size]
        batch_fast = fast_periods[i:i + batch_size]
        batch_med = med_periods[i:i + batch_size]
        batch_slow = slow_periods[i:i + batch_size]
        
        # Extract batch parameters for enabled indicators
        if USE_VOLUME_FILTER:
            batch_vol_windows = vol_windows[i:i + batch_size]
            batch_vol_mults = vol_mults[i:i + batch_size]
        if USE_RSI:
            batch_rsi_windows = rsi_windows[i:i + batch_size]
            batch_rsi_threshold = rsi_thresholds[i:i + batch_size]
        
        # Validate window sizes
        if any(w <= 0 for w in batch_fast + batch_med + batch_slow):
            continue
        
        # Calculate EMAs for the batch
        ema_f = vbt.MA.run(train_close, window=batch_fast, ewm=True).ma
        ema_m = vbt.MA.run(train_close, window=batch_med, ewm=True).ma
        ema_s = vbt.MA.run(train_close, window=batch_slow, ewm=True).ma
        
        # Set column names to combo tuples for tracking
        ema_f.columns = batch_combos
        ema_m.columns = batch_combos
        ema_s.columns = batch_combos
        
        # Base trend condition: EMAs in bullish alignment
        trend = (ema_f > ema_m) & (ema_m > ema_s)
        entries = trend.copy()
        
        # --- VOLUME FILTER ---
        if USE_VOLUME_FILTER:
            vol_ma = vbt.MA.run(train_volume, window=batch_vol_windows).ma
            vol_ma.columns = batch_combos
            vol_raw = pd.DataFrame(
                np.tile(train_volume.values.reshape(-1, 1), len(batch_combos)),
                index=train_volume.index,
                columns=batch_combos
            )
            mult_arr = pd.DataFrame(
                np.tile(batch_vol_mults, (len(train_volume), 1)),
                index=train_volume.index,
                columns=batch_combos
            )
            vol_condition = vol_raw > (vol_ma * mult_arr)
            entries = entries & vol_condition
        
        # --- RSI FILTER ---
        if USE_RSI:
            rsi = vbt.RSI.run(train_close, window=batch_rsi_windows).rsi
            rsi.columns = batch_combos
            rsi_thresh_arr = pd.DataFrame(
                np.tile(batch_rsi_threshold, (len(train_close), 1)),
                index=train_close.index,
                columns=batch_combos
            )
            rsi_condition = rsi > rsi_thresh_arr
            entries = entries & rsi_condition
        
        # EXIT SIGNAL: Fast EMA crosses below Medium EMA
        exits = (ema_f < ema_m)
        
        # Clean signals
        entries = entries.vbt.signals.first(after=exits)
        
        # RUN PORTFOLIO SIMULATION
        pf = vbt.Portfolio.from_signals(
            train_close,
            entries,
            exits,
            init_cash=init_cash,
            fees=fees,
            freq='D'
        )
        
        # Store results
        all_sharpe.extend(pf.sharpe_ratio().values)
        all_returns.extend(pf.total_return().values)
        all_combos.extend(batch_combos)
    
    # Find best strategy by Sharpe ratio for this asset
    all_sharpe_clean = [s if not np.isnan(s) else -np.inf for s in all_sharpe]
    best_idx = np.argmax(all_sharpe_clean)
    best_params_tuple = all_combos[best_idx]
    best_sharpe = all_sharpe[best_idx]
    best_return = all_returns[best_idx]
    
    # Convert tuple to dictionary
    best_params = extract_best_params(best_params_tuple)
    
    # Store best parameters for this asset
    best_params_per_asset[asset_name] = {
        'params': best_params,
        'sharpe': best_sharpe,
        'return': best_return,
        'params_tuple': best_params_tuple
    }
    
    # Print results
    print(f"  Best Params: fast_ema={best_params['fast_ema']}, med_ema={best_params['med_ema']}, slow_ema={best_params['slow_ema']}", end="")
    if USE_VOLUME_FILTER:
        print(f", vol_window={best_params['vol_window']}, vol_mult={best_params['vol_mult']:.2f}", end="")
    print()
    print(f"  Train Sharpe: {best_sharpe:.2f}")
    print(f"  Train Return: {best_return*100:.1f}%")

print()
print("=" * 50)
print("✅ Multi-Asset Grid Search Complete!")
print()
print("📊 Summary of Best Parameters:")
for asset_name in ASSET_NAMES:
    info = best_params_per_asset[asset_name]
    print(f"  {asset_name}: Sharpe={info['sharpe']:.2f}, Return={info['return']*100:.1f}%")

In [ ]:
# =============================================================================
# CELL 5 - MOMENTUM SCORING SYSTEM
# =============================================================================

print("📊 Calculating Momentum Scores")
print("=" * 50)

# Calculate momentum scores for validation period
momentum_scores = pd.DataFrame(index=val_data[ASSET_NAMES[0]]['close'].index)
momentum_components = {}

for asset_name in ASSET_NAMES:
    params = best_params_per_asset[asset_name]['params']
    val_close = val_data[asset_name]['close']
    val_volume = val_data[asset_name]['volume']
    
    # Calculate momentum score
    score, components = calculate_momentum_score(val_close, val_volume, params)
    momentum_scores[asset_name] = score
    momentum_components[asset_name] = components
    
    print(f"{asset_name}: Mean Score = {score.mean():.1f}, Max = {score.max():.1f}")

# Determine selected asset for each day
def select_asset(row):
    max_score = row.max()
    if max_score < CASH_THRESHOLD:
        return 'CASH'
    return row.idxmax()

momentum_scores['Selected_Asset'] = momentum_scores[ASSET_NAMES].apply(select_asset, axis=1)

# Apply momentum lookback smoothing if configured
if MOMENTUM_LOOKBACK > 1:
    for asset_name in ASSET_NAMES:
        momentum_scores[f"{asset_name}_smoothed"] = momentum_scores[asset_name].rolling(MOMENTUM_LOOKBACK).mean()

print()
print("📋 Sample Momentum Scores (first 10 days):")
print(momentum_scores[ASSET_NAMES + ['Selected_Asset']].head(10).to_string())

print()
print("📊 Asset Selection Distribution:")
selection_counts = momentum_scores['Selected_Asset'].value_counts()
for asset, count in selection_counts.items():
    pct = count / len(momentum_scores) * 100
    print(f"  {asset}: {count} days ({pct:.1f}%)")

print()
print("✅ Momentum scoring complete!")

In [ ]:
# =============================================================================
# CELL 6 - VALIDATION BACKTEST (Rotation Strategy)
# =============================================================================

print("📈 Running Validation Backtest")
print("=" * 50)

# Get validation date range
val_dates = val_data[ASSET_NAMES[0]]['close'].index

# Initialize portfolio tracking
portfolio_value = [init_cash]
daily_returns = []
positions = []  # Track which asset is held each day
switches = 0
current_position = None

# Get daily returns for each asset
asset_returns = {}
for asset_name in ASSET_NAMES:
    asset_returns[asset_name] = val_data[asset_name]['close'].pct_change().fillna(0)

# Run the rotation strategy
for i, date in enumerate(val_dates):
    selected = momentum_scores.loc[date, 'Selected_Asset']
    
    # Track position changes
    if current_position != selected:
        if current_position is not None:
            switches += 1
        current_position = selected
    
    positions.append(selected)
    
    # Calculate daily return based on position
    if selected == 'CASH':
        daily_ret = 0  # Cash earns 0 return
    else:
        daily_ret = asset_returns[selected].loc[date]
        # Apply transaction fees on switches
        if i > 0 and positions[i-1] != selected and positions[i-1] != 'CASH' and selected != 'CASH':
            daily_ret -= fees * 2  # Buy and sell fees
    
    daily_returns.append(daily_ret)
    new_value = portfolio_value[-1] * (1 + daily_ret)
    portfolio_value.append(new_value)

# Remove initial value
portfolio_value = portfolio_value[1:]

# Create portfolio series
portfolio_series = pd.Series(portfolio_value, index=val_dates)
returns_series = pd.Series(daily_returns, index=val_dates)

# Calculate metrics
total_return = (portfolio_value[-1] / init_cash - 1) * 100
sharpe_ratio = returns_series.mean() / returns_series.std() * np.sqrt(252) if returns_series.std() > 0 else 0

# Calculate drawdown
cummax = portfolio_series.cummax()
drawdown = (portfolio_series - cummax) / cummax * 100
max_drawdown = drawdown.min()

# Win rate (days with positive returns when in position)
in_position_returns = [r for r, p in zip(daily_returns, positions) if p != 'CASH']
win_rate = sum(1 for r in in_position_returns if r > 0) / len(in_position_returns) * 100 if in_position_returns else 0

# Asset allocation breakdown
position_counts = pd.Series(positions).value_counts()

# Calculate buy-and-hold benchmarks
benchmarks = {}
for asset_name in ASSET_NAMES:
    start_price = val_data[asset_name]['close'].iloc[0]
    end_price = val_data[asset_name]['close'].iloc[-1]
    benchmarks[asset_name] = (end_price / start_price - 1) * 100

# Print results
print()
print("📊 VALIDATION RESULTS (Rotation Strategy)")
print("=" * 50)
print(f"Period: {val_dates[0].date()} to {val_dates[-1].date()}")
print(f"Total Return: {total_return:+.1f}%")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.1f}%")
print(f"Win Rate: {win_rate:.1f}%")
print()
print("Asset Allocation:")
for asset in ASSET_NAMES + ['CASH']:
    count = position_counts.get(asset, 0)
    pct = count / len(positions) * 100
    print(f"  {asset}: {pct:.1f}% of time")
print()
print(f"Number of Asset Switches: {switches}")
print()
print("📈 Buy-and-Hold Benchmarks:")
for asset_name in ASSET_NAMES:
    print(f"  {asset_name}: {benchmarks[asset_name]:+.1f}%")

# Store results for visualization
rotation_results = {
    'portfolio_value': portfolio_series,
    'returns': returns_series,
    'positions': pd.Series(positions, index=val_dates),
    'total_return': total_return,
    'sharpe': sharpe_ratio,
    'max_drawdown': max_drawdown,
    'win_rate': win_rate,
    'switches': switches,
    'benchmarks': benchmarks
}

print()
print("✅ Validation backtest complete!")

In [ ]:
# =============================================================================
# CELL 7 - VISUALIZATION
# =============================================================================

print("📊 Generating Visualizations")
print("=" * 50)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# =================================================================
# Plot 1: Cumulative Returns Comparison
# =================================================================
ax1 = axes[0, 0]

# Normalize all to start at 100
rotation_norm = (rotation_results['portfolio_value'] / init_cash) * 100
ax1.plot(rotation_norm.index, rotation_norm.values, label='Rotation Strategy', linewidth=2, color='blue')

colors = {'TQQQ': 'green', 'GLD': 'gold', 'BTC': 'orange'}
for asset_name in ASSET_NAMES:
    bh_series = (val_data[asset_name]['close'] / val_data[asset_name]['close'].iloc[0]) * 100
    ax1.plot(bh_series.index, bh_series.values, label=f'{asset_name} B&H', 
             linewidth=1.5, alpha=0.7, color=colors.get(asset_name, 'gray'))

ax1.axhline(y=100, color='black', linestyle='--', alpha=0.3)
ax1.set_title('Cumulative Returns Comparison', fontsize=14, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Value (starting at 100)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# =================================================================
# Plot 2: Momentum Scores Over Time
# =================================================================
ax2 = axes[0, 1]

for asset_name in ASSET_NAMES:
    ax2.plot(momentum_scores.index, momentum_scores[asset_name].values, 
             label=asset_name, alpha=0.8, color=colors.get(asset_name, 'gray'))

ax2.axhline(y=CASH_THRESHOLD, color='red', linestyle='--', alpha=0.7, label=f'Cash Threshold ({CASH_THRESHOLD})')
ax2.set_title('Momentum Scores Over Time', fontsize=14, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Momentum Score (0-100)')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 100)

# =================================================================
# Plot 3: Asset Allocation Timeline
# =================================================================
ax3 = axes[1, 0]

# Create color map for positions
position_colors = {'TQQQ': 0, 'GLD': 1, 'BTC': 2, 'CASH': 3}
position_values = [position_colors.get(p, 3) for p in rotation_results['positions']]

# Create scatter plot for position timeline
scatter = ax3.scatter(rotation_results['positions'].index, position_values, 
                       c=position_values, cmap='Set1', s=5, alpha=0.7)

ax3.set_yticks([0, 1, 2, 3])
ax3.set_yticklabels(['TQQQ', 'GLD', 'BTC', 'CASH'])
ax3.set_title('Asset Allocation Timeline', fontsize=14, fontweight='bold')
ax3.set_xlabel('Date')
ax3.set_ylabel('Position')
ax3.grid(True, alpha=0.3)

# =================================================================
# Plot 4: Drawdown Comparison
# =================================================================
ax4 = axes[1, 1]

# Rotation strategy drawdown
cummax_rot = rotation_results['portfolio_value'].cummax()
dd_rot = (rotation_results['portfolio_value'] - cummax_rot) / cummax_rot * 100
ax4.fill_between(dd_rot.index, dd_rot.values, 0, alpha=0.3, color='blue', label='Rotation Strategy')
ax4.plot(dd_rot.index, dd_rot.values, color='blue', linewidth=1)

# Buy-and-hold drawdowns
for asset_name in ASSET_NAMES:
    bh_series = val_data[asset_name]['close']
    cummax_bh = bh_series.cummax()
    dd_bh = (bh_series - cummax_bh) / cummax_bh * 100
    ax4.plot(dd_bh.index, dd_bh.values, label=f'{asset_name} B&H', 
             alpha=0.7, linewidth=1, color=colors.get(asset_name, 'gray'))

ax4.set_title('Drawdown Comparison', fontsize=14, fontweight='bold')
ax4.set_xlabel('Date')
ax4.set_ylabel('Drawdown (%)')
ax4.legend(loc='lower left')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rotation_strategy_results.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print("✅ Visualizations saved to 'rotation_strategy_results.png'")

# =================================================================
# Summary Table
# =================================================================
print()
print("📋 STRATEGY COMPARISON SUMMARY")
print("=" * 60)
print(f"{'Strategy':<20} {'Return':>12} {'Sharpe':>10} {'Max DD':>10}")
print("-" * 60)
print(f"{'Rotation':<20} {rotation_results['total_return']:>+11.1f}% {rotation_results['sharpe']:>10.2f} {rotation_results['max_drawdown']:>9.1f}%")

for asset_name in ASSET_NAMES:
    bh_series = val_data[asset_name]['close']
    bh_returns = bh_series.pct_change().fillna(0)
    bh_sharpe = bh_returns.mean() / bh_returns.std() * np.sqrt(252) if bh_returns.std() > 0 else 0
    cummax_bh = bh_series.cummax()
    dd_bh = ((bh_series - cummax_bh) / cummax_bh * 100).min()
    bh_total_return = rotation_results['benchmarks'][asset_name]
    print(f"{asset_name + ' B&H':<20} {bh_total_return:>+11.1f}% {bh_sharpe:>10.2f} {dd_bh:>9.1f}%")

print("=" * 60)